In [1]:
import os

os.environ["GRB_LICENSE_FILE"] = r"C:\Users\PC3\Desktop\gurobi.lic"

import gurobipy as gp
from gurobipy import GRB 


# Data Builder

In [5]:
from dataclasses import replace
import importlib
import uc_experiment_builders
importlib.reload(uc_experiment_builders)

from uc_experiment_builders import (
    load_network_uc_from_excel_fixed,   # or build_data_ieee_case14 if using IEEE
    diversify_demand_archetypes_preserve_C,
    scale_renewables,
    make_scenario_data,
)

# ============================================================
# 0) Build base_data ONCE (run this cell after defining paths)
# ============================================================
# Example: Gabriel Excel dataset
XLSX_PATH = r"C:\Users\PC3\Desktop\Counterfactual-Explanations-for-optimization-problems\Mixed\UC-Experiments\RedEjemploRiesgoConfiabilidad.xlsx"  # <-- put your path

base_data = load_network_uc_from_excel_fixed(
    xlsx_path=XLSX_PATH,
    wind_scenario=2,
    solar_scenario="Alto",
    curt_penalty=1000.0,
    allow_shifting=False,
    carbon_price=0.0,
    slack_bus=0,
    rep_params=None,                      # <-- you already have these in notebook
    emission_rates=None,     # <-- you already have these
)

# ============================================================
# 1) Scenario knobs
# ============================================================
ARCHETYPE_SEED = 7
REN_SCALE = 0.0

# ============================================================
# 2) Build scenario data from base_data
# ============================================================
data, types = make_scenario_data(base_data, archetype_seed=ARCHETYPE_SEED, ren_scale=REN_SCALE)

print("Scenario ready:")
print("  ARCHETYPE_SEED =", ARCHETYPE_SEED)
print("  REN_SCALE      =", REN_SCALE)
print("  demand shape   =", data.demand.shape)
print("  #gens, #rens   =", len(data.gens), len(data.rens))


Scenario ready:
  ARCHETYPE_SEED = 7
  REN_SCALE      = 0.0
  demand shape   = (14, 24)
  #gens, #rens   = 6 4


In [6]:
from b3_ncxplain import run_B3_ncxplain_shift_prices, foil_force_unit_commitment

g_foil = 2   # 0-based generator index
t_foil = 10  # 0..T-1
window_size = 24
per_bus_neutrality = True
foil_fn = foil_force_unit_commitment(g=g_foil, t=t_foil, on=False)  # force OFF at hour t

out = run_B3_ncxplain_shift_prices(
    data=data,
    window_size=window_size,
    per_bus_neutrality=per_bus_neutrality,
    alpha= 1,                 # e.g. curtailment <= 0.8 * factual
    foil_fn=None,           # extra foil constraint
    price_lb=-200.0,
    price_ub=200.0,
    curt_lb=0.0,
    curt_ub=1000.0,
    max_iters=30,
    mp_time_limit=60.0,
    output_flag_mp=1,
    output_flag_sp=0,
)
out["status"], out.get("iters"), out.get("gap")


Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 60
Set parameter NumericFocus to value 2
Set parameter InfUnbdInfo to value 1
Set parameter DualReductions to value 0
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-14700, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 28 logical processors, using up to 28 threads

Non-default parameters:
TimeLimit  60
DualReductions  0
InfUnbdInfo  1
NumericFocus  2

Academic license 2653481 - for non-commercial use only - registered to to___@ug.uchile.cl
Optimize a model with 1537 rows, 1536 columns and 3072 nonzeros (Min)
Model fingerprint: 0x569dbfb6
Model has 768 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [2e+02, 1e+03]
  RHS range        [1e+03, 1e+03]

Presolve time: 0.01s
Presolved: 768 rows, 1536 columns, 1536 nonzeros

Iteration    Objective   

('OPTIMAL', 1, 0.0)

In [7]:
print(out)

{'status': 'OPTIMAL', 'iters': 1, 'C_factual': 0.0, 'C_bar': 0.0, 'C_foil': 0.0, 'C_opt_under_new_coeffs': 0.0, 'pi_plus_base': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0